# Ultra-FineWeb Boundary Quality Review

## tl;dr

唯一质量指标是目标断点是否合理。复用冻结的 640 条盲审记录：637 条判为合理，0 条明确不合理，3 条不确定。目前无法区分不同分数段的断点质量，撤回优先选择 0.99 的建议。不修改原始标注，不把文字瑕疵或文体计作断点错误，没有新增样本或训练。

## Context & Methods

全量分数普查与分层抽样分开。640 条语义判断在隐藏分数时完成，44 条初始异常／不确定项及随机 80 条 A 类复核后冻结。AI 初审，不是人工金标准。样本抽取执行当前 v8 清理与最短边至少两字。

### Key Assumptions

每篇符合条件的训练文档随机取一个切分题，因此语义率是文档等权口径，不是训练行等权；不跨分数段直接平均。文档等权产出与训练行标点分布分别计算。事实真伪未评估。

build.py 顺序执行下列单元并捕获输出，未使用 Jupyter 内核。依赖 Python 3 和已有 training/.deps；运行时从仓库根目录开始。

In [1]:
from pathlib import Path
import collections, hashlib, json, math, re, statistics, sys, unicodedata
root = Path.cwd()
while not (root/'training/reviews/ultra-score-review-20261004').exists():
    if root.parent == root: raise RuntimeError('Run inside repository')
    root = root.parent
review = root/'training/reviews/ultra-score-review-20261004'
out = root/'training/artifacts/ultra-score-review-20261004'
sys.path[:0] = [str(root/'training/.deps'), str(root/'training')]
from prepare_synthetic_data import clean_document, is_han
def read(name): return json.loads((review/name).read_text())
freeze = read('freeze.json')
for name, expected in freeze['sha256'].items():
    assert hashlib.sha256((review/name).read_bytes()).hexdigest() == expected, name
for entry in read('evidence/index.json'):
    assert hashlib.sha256((review/'evidence'/entry['file']).read_bytes()).hexdigest() == entry['sha256']
profile, protocol = read('profile.json'), read('protocol.json')
samples, annotations = read('sample.json'), read('annotations.json')
documents_path = out/'documents.jsonl'
documents = [json.loads(line) for line in documents_path.open()]
assert len(documents) == 4800 and len(samples) == len(annotations) == 640
assert len({r['document_id'] for r in samples}) == 640
docs_by_location = {(d['raw_file'], d['group'], d['row']):d for d in documents}
joined = []
for s,a in zip(samples,annotations):
    assert s['review_id'] == a['review_id']
    d = docs_by_location[s['raw_file'],s['group'],s['row']]
    assert d['split']=='train' and d['quality_document']
    assert d['content_sha256']==hashlib.sha256(d['content'].encode()).hexdigest()
    assert any(p['text']==s['text'] and p['cut']==s['cut'] for p in d['pairs'])
    assert min(s['cut'],s['length']-s['cut']) >= 2
    joined.append(dict(**s,**{k:v for k,v in a.items() if k!='review_id'}))
assert sum(sum(h) for h in profile['histograms'].values())==profile['rows']==131006462
assert sum(f['rows'] for f in profile['files'])==profile['rows']
assert sum(g['rows'] for g in protocol['groups'])==protocol['frame_rows']==1700594
print('Verified 256-file census, 4,800 sampled documents and 640 frozen, distinct train-only reviews.')
print('Raw contents hash:',hashlib.sha256(documents_path.read_bytes()).hexdigest())


Verified 256-file census, 4,800 sampled documents and 640 frozen, distinct train-only reviews.
Raw contents hash: 86e521b4f20edd45ec6bd403daa12e6bc8b88299ef6d46f14dc477f5b1617295


## Supporting diagnostics — not the decision metric

保留先前的文本文字诊断供追溯，不参与此次断点质量排序。全量文档数精确计数，未应用本地清理／去重。分层率的分母为该组抽查的 80 个独立文档，每篇一个题；Wilson 区间仅是独立二项近似参考，不包含抽样簇、标注误差或多重比较校正，不据此宣称显著胜出。

In [2]:
def wilson(k,n):
    z=1.959963984540054; p=k/n; den=1+z*z/n
    center=(p+z*z/(2*n))/den
    half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return center-half, center+half
thresholds=[]
source_composition=[]
for t in profile['thresholds']:
    tick=round(t['threshold']*10000)
    assert sum(sum(h[tick:]) for h in profile['histograms'].values())==t['documents']
    assert sum(t['sources'].values())==t['documents']
    label='≥ '+str(t['threshold']) if tick<10000 else '= 1.0'
    thresholds.append(dict(threshold=label, score=t['threshold'], documents=t['documents'],
                           retained=f"{100*t['fraction']:.3f}%", share=t['fraction']))
    for source,n in t['sources'].items():
        source_composition.append(dict(threshold=label,source=source,documents=n,share=n/t['documents']))
bands=[]
source_review=[]
for b,label in enumerate(protocol['strata']):
    rows=[r for r in joined if r['bucket']==b]; assert len(rows)==80
    counts=collections.Counter(r['code'] for r in rows)
    damaged=sum(r['text_type']=='damaged' for r in rows)
    structural=sum(r['severity']=='structural_or_severe' for r in rows)
    lo,hi=wilson(damaged,len(rows))
    bands.append(dict(band=label,n=len(rows),damage=damaged,damage_rate=damaged/len(rows),
                      damage_percent=f'{100*damaged/len(rows):.2f}%',
                      wilson_reference=f'{100*lo:.1f}–{100*hi:.1f}%',
                      structural=structural,minor=damaged-structural,boilerplate=counts['B'],
                      boundary_error=counts['E'],boundary_uncertain=counts['X']+counts['U'],
                      median_length=statistics.median(r['length'] for r in rows)))
    for source in sorted({r['source'] for r in rows}):
        ss=[r for r in rows if r['source']==source]
        source_review.append(dict(band=label,source=source,n=len(ss),
                                 damage=sum(r['text_type']=='damaged' for r in ss)))
print(json.dumps(bands,ensure_ascii=False,indent=2))
print('No confirmed wrong boundary labels in this review; uncertain:',sum(r['boundary_uncertain'] for r in bands))
print('Zero out of 80 Wilson upper reference:',wilson(0,80)[1])


[
  {
    "band": "[0.5,0.8)",
    "n": 80,
    "damage": 5,
    "damage_rate": 0.0625,
    "damage_percent": "6.25%",
    "wilson_reference": "2.7–13.8%",
    "structural": 4,
    "minor": 1,
    "boilerplate": 2,
    "boundary_error": 0,
    "boundary_uncertain": 0,
    "median_length": 32.0
  },
  {
    "band": "[0.8,0.9)",
    "n": 80,
    "damage": 2,
    "damage_rate": 0.025,
    "damage_percent": "2.50%",
    "wilson_reference": "0.7–8.7%",
    "structural": 1,
    "minor": 1,
    "boilerplate": 0,
    "boundary_error": 0,
    "boundary_uncertain": 0,
    "median_length": 31.5
  },
  {
    "band": "[0.9,0.95)",
    "n": 80,
    "damage": 5,
    "damage_rate": 0.0625,
    "damage_percent": "6.25%",
    "wilson_reference": "2.7–13.8%",
    "structural": 4,
    "minor": 1,
    "boilerplate": 2,
    "boundary_error": 0,
    "boundary_uncertain": 2,
    "median_length": 33.0
  },
  {
    "band": "[0.95,0.98)",
    "n": 80,
    "damage": 4,
    "damage_rate": 0.05,
    "damage_percent

### 抽取适配性与明确的表面指标

以下为 4,800 篇样本中训练划分的文档，含未通过本地过滤者。quality_document 要求汉字 ≥80 且 汉字/(汉字+ASCII 字母) ≥0.70；它不是人工质量分。产出只做文档内同文同切点去重，未做历史 Bloom、跨文档或近重复去重。字面转义和半角逗号只是可复算的表面指标，不自动等同于错误标签。

In [3]:
patterns=protocol['policy']['sample_filter']['fragment_patterns']
literal=re.compile('|'.join(patterns[k] for k in ['literal_line_escape','literal_control_escape','literal_unicode_residue']))
extraction=[]
for b,label in enumerate(protocol['strata']):
    ds=[d for d in documents if d['bucket']==b and d['split']=='train']
    assert len([d for d in documents if d['bucket']==b])==600
    ps=[p for d in ds for p in d['pairs']]
    eligible=sum(bool(d['pairs']) for d in ds)
    lit=sum(bool(literal.search(d['content'])) for d in ds)
    ascii_only=sum(',' in d['content'] and '，' not in d['content'] for d in ds)
    short=latin=0
    for d in ds:
        text=unicodedata.normalize('NFKC',clean_document(d['content']))
        han=sum(is_han(c) for c in text)
        lat=sum(c.isascii() and c.isalpha() for c in text)
        short+=han<80; latin+=han/max(1,han+lat)<.70
    extraction.append(dict(band=label,train_documents=len(ds),eligible=eligible,
        eligible_rate=eligible/len(ds),eligible_percent=f'{100*eligible/len(ds):.1f}%',
        quality_reject=sum(not d['quality_document'] for d in ds),short_han=short,low_han_ratio=latin,
        no_pair_after_quality=sum(d['quality_document'] and not d['pairs'] for d in ds),
        pairs=len(ps),mean_pairs=round(len(ps)/len(ds),2),
        literal_escape=lit,literal_rate=lit/len(ds),ascii_comma_only=ascii_only,
        ascii_comma_percent=f'{100*ascii_only/len(ds):.1f}%',
        median_length=statistics.median(p['length'] for p in ps),
        period_share=sum(p['punctuation']=='。' for p in ps)/len(ps),
        review_period_share=sum(r['punctuation']=='。' for r in joined if r['bucket']==b)/80))
    assert eligible+extraction[-1]['quality_reject']+extraction[-1]['no_pair_after_quality']==len(ds)
print(json.dumps(extraction,ensure_ascii=False,indent=2))
print('Sample duplicate Han signatures:',len(documents)-len({d['document_id'] for d in documents}))
print('Sample duplicate exact content:',len(documents)-len({d['content_sha256'] for d in documents}))


[
  {
    "band": "[0.5,0.8)",
    "train_documents": 575,
    "eligible": 344,
    "eligible_rate": 0.5982608695652174,
    "eligible_percent": "59.8%",
    "quality_reject": 42,
    "short_han": 19,
    "low_han_ratio": 33,
    "no_pair_after_quality": 189,
    "pairs": 17070,
    "mean_pairs": 29.69,
    "literal_escape": 1,
    "literal_rate": 0.0017391304347826088,
    "ascii_comma_only": 280,
    "ascii_comma_percent": "48.7%",
    "median_length": 23.0,
    "period_share": 0.2813122437024019,
    "review_period_share": 0.5125
  },
  {
    "band": "[0.8,0.9)",
    "train_documents": 570,
    "eligible": 312,
    "eligible_rate": 0.5473684210526316,
    "eligible_percent": "54.7%",
    "quality_reject": 55,
    "short_han": 10,
    "low_han_ratio": 52,
    "no_pair_after_quality": 203,
    "pairs": 8885,
    "mean_pairs": 15.59,
    "literal_escape": 1,
    "literal_rate": 0.0017543859649122807,
    "ascii_comma_only": 319,
    "ascii_comma_percent": "56.0%",
    "median_length": 

### 稳健性与边界

避免用“所有高分段合并 160 个样本”的简单平均估全库：各分层人口和可抽取率不同。分来源复查仅描述，不将来源命名为领域，也不引入来源权重。没有运行模型，因此不能输出准确率增益或最优阈值。

In [4]:
sensitivity=[]
for label in protocol['strata']:
    rr=[r for r in joined if protocol['strata'][r['bucket']]==label and r['source']=='Tele']
    k=sum(r['text_type']=='damaged' for r in rr)
    sensitivity.append(dict(band=label,source='Tele',n=len(rr),damage=k,rate=k/len(rr)))
print('Same-source diagnostic (small n; not causality):',json.dumps(sensitivity,ensure_ascii=False))
print('Structural/severe-only counts by ascending band:',[r['structural'] for r in bands])
assert sum(r['damage'] for r in bands)==40
assert sum(r['boundary_error'] for r in bands)==0
assert sum(r['boundary_uncertain'] for r in bands)==3
print('Key count reconciliation passed. Training remains paused.')


Same-source diagnostic (small n; not causality): [{"band": "[0.5,0.8)", "source": "Tele", "n": 31, "damage": 1, "rate": 0.03225806451612903}, {"band": "[0.8,0.9)", "source": "Tele", "n": 31, "damage": 1, "rate": 0.03225806451612903}, {"band": "[0.9,0.95)", "source": "Tele", "n": 28, "damage": 0, "rate": 0.0}, {"band": "[0.95,0.98)", "source": "Tele", "n": 42, "damage": 1, "rate": 0.023809523809523808}, {"band": "[0.98,0.99)", "source": "Tele", "n": 40, "damage": 1, "rate": 0.025}, {"band": "[0.99,0.999)", "source": "Tele", "n": 44, "damage": 1, "rate": 0.022727272727272728}, {"band": "[0.999,1)", "source": "Tele", "n": 44, "damage": 5, "rate": 0.11363636363636363}, {"band": "1.0", "source": "Tele", "n": 36, "damage": 1, "rate": 0.027777777777777776}]
Structural/severe-only counts by ascending band: [4, 1, 4, 3, 4, 2, 7, 4]
Key count reconciliation passed. Training remains paused.


## Decision metric — boundary acceptability only

读取冻结标注中独立的 boundary 字段，A 为合理、E 为明确不合理、U 为不确定。初始代码 B（附属文字）和 D（文字有瑕疵）若 boundary=A，仍完整计入合理。不得因文字问题扣分，不确定不得计作正确或错误。一个位置可合理而非唯一最优；原文有标点也不是合理的充分依据。下表使用所有 80 个样本作分母，并保留未决数量。

In [5]:
boundary_bands=[]
for b,label in enumerate(protocol['strata']):
    rows=[r for r in joined if r['bucket']==b]
    c=collections.Counter(r['boundary'] for r in rows)
    assert set(c) <= {'A','E','U'}
    assert c['A']+c['E']+c['U']==len(rows)==80
    boundary_bands.append(dict(band=label,n=len(rows),acceptable=c['A'],
        wrong=c['E'],uncertain=c['U'],acceptable_count=f"{c['A']}/{len(rows)}",
        wrong_count=f"{c['E']}/{len(rows)}",acceptable_share=c['A']/len(rows)))
assert sum(r['acceptable'] for r in boundary_bands)==637
assert sum(r['wrong'] for r in boundary_bands)==0
assert sum(r['uncertain'] for r in boundary_bands)==3
assert all(r['boundary']=='A' for r in joined if r['code'] in ['B','D'])
print(json.dumps(boundary_bands,ensure_ascii=False,indent=2))
print('Uncertain IDs:',[r['review_id'] for r in joined if r['boundary']=='U'])
print('Previously flagged text-damage rows still accepted:',sum(r['text_type']=='damaged' and r['boundary']=='A' for r in joined))
print('Reference only: zero errors among 80 independent fully adjudicated examples gives Wilson upper bound',wilson(0,80)[1])


[
  {
    "band": "[0.5,0.8)",
    "n": 80,
    "acceptable": 80,
    "wrong": 0,
    "uncertain": 0,
    "acceptable_count": "80/80",
    "wrong_count": "0/80",
    "acceptable_share": 1.0
  },
  {
    "band": "[0.8,0.9)",
    "n": 80,
    "acceptable": 80,
    "wrong": 0,
    "uncertain": 0,
    "acceptable_count": "80/80",
    "wrong_count": "0/80",
    "acceptable_share": 1.0
  },
  {
    "band": "[0.9,0.95)",
    "n": 80,
    "acceptable": 78,
    "wrong": 0,
    "uncertain": 2,
    "acceptable_count": "78/80",
    "wrong_count": "0/80",
    "acceptable_share": 0.975
  },
  {
    "band": "[0.95,0.98)",
    "n": 80,
    "acceptable": 80,
    "wrong": 0,
    "uncertain": 0,
    "acceptable_count": "80/80",
    "wrong_count": "0/80",
    "acceptable_share": 1.0
  },
  {
    "band": "[0.98,0.99)",
    "n": 80,
    "acceptable": 80,
    "wrong": 0,
    "uncertain": 0,
    "acceptable_count": "80/80",
    "wrong_count": "0/80",
    "acceptable_share": 1.0
  },
  {
    "band": "[0.99,0.9

## Takeaways

所有分数段均未发现明确错误断点；这表示当前初审无法区分，并非证明各段等质或真实错误率为零。撤回优先选择 0.99 的建议，不按文字瑕疵率筛阈值。下一轮若扩大研究，应继续以断点合理性为唯一指标，并预先冻结判定标准、隐藏 score、保留不确定类、增加独立复核及相同口径的对照样本。现有数据和训练配置不变，训练仍暂停。